# Data cleaning

Nine source files from the Olist Brazilian e-commerce dataset, cleaned one at a time into a `clean` schema. Each section states the grain of the source table, examines it, records what is wrong with it, and builds a cleaned version.

## Summary of what the cleaning found

Five issues change how the data must be analysed, rather than merely needing a fix:

1. **The promised delivery date carries no time of day.** Every estimate is stored at midnight, so comparing it to a real delivery timestamp counts 1,292 same-day deliveries as late. The late rate is 6.77% measured in whole days and 8.11% measured as timestamps.
2. **Most reviews on late orders were written before the parcel arrived** - 83.6% of them, against 3.3% on on-time orders. The review survey went out on the estimated delivery date. Those reviews average 1.98 stars; late orders reviewed after arrival average 3.78.
3. **Payment approval is a real delay for one payment method.** Credit cards clear in 16 minutes at the median, boleto bank slips in 29 hours.
4. **1,382 orders have timestamps in an impossible order**, mostly carrier pickup recorded before payment approval.
5. **Two revenue figures exist and they disagree by 165,319 BRL.** The reconciliation is in section 8.

## Approach

- `raw` is never modified. Every cleaned object is new, so any figure can be traced back to the untouched file.
- One clean table per source file, at the source's own grain. Joining and aggregating is modelling and happens in the next notebook; mixing the two loses track of what a row represents.
- Rows that look wrong are flagged, not deleted. Deleting changes every downstream count silently; a flag lets each analysis decide.
- Every decision is recorded with the number of rows it affects.
- Each table is verified after cleaning: row count, key uniqueness, and the specific thing that was changed.

## Order of work

Lookups first, then the tables that depend on them.

| # | Table | Depends on |
| --- | --- | --- |
| 1 | geolocation | - (builds the zip lookup used by customers and sellers) |
| 2 | customers | geolocation |
| 3 | sellers | geolocation |
| 4 | category_translation | - |
| 5 | products | category_translation |
| 6 | orders | - |
| 7 | order_items | - |
| 8 | order_payments | - |
| 9 | order_reviews | - |

In [22]:
import duckdb

con = duckdb.connect("../olist.duckdb")
con.sql("CREATE SCHEMA IF NOT EXISTS clean")

def show(sql, n=20):
    """Run a query and print it."""
    con.sql(sql).show(max_rows=n)

def check(label, sql, expected=None):
    """Run a one-value query and print it, with a pass/fail if an expectation is given."""
    got = con.execute(sql).fetchall()[0][0]
    if expected is None:
        print(f"{label}: {got:,}" if isinstance(got, int) else f"{label}: {got}")
    else:
        flag = "OK" if got == expected else "MISMATCH"
        print(f"{label}: {got:,} (expected {expected:,}) {flag}")

print("connected")

connected


---

## 1. geolocation

**Grain:** one coordinate point within a zip prefix - not one row per zip, which is the central problem with this file.

**Purpose:** produce one latitude and longitude per zip prefix so that seller-to-customer distance can be measured, and a reliable city and state per zip, since the location text typed into the sellers file is unusable (section 3).

The checks below measure how many points each zip carries, whether any coordinates fall outside Brazil, and whether a zip ever contradicts itself about its own city or state.

In [23]:
show("""
    SELECT
        COUNT(*)                                        AS rows,
        COUNT(DISTINCT geolocation_zip_code_prefix)     AS zips,
        ROUND(COUNT(*) * 1.0
              / COUNT(DISTINCT geolocation_zip_code_prefix), 1) AS avg_points_per_zip
    FROM raw.geolocation
""")

# exact duplicate rows
show("""
    SELECT SUM(c - 1) AS duplicate_rows
    FROM (
        SELECT geolocation_zip_code_prefix, geolocation_lat, geolocation_lng,
               geolocation_city, geolocation_state, COUNT(*) AS c
        FROM raw.geolocation
        GROUP BY ALL
        HAVING c > 1
    )
""")

# coordinates outside Brazil's bounding box
show("""
    SELECT COUNT(*) AS outside_brazil
    FROM raw.geolocation
    WHERE CAST(geolocation_lat AS DOUBLE) NOT BETWEEN -34 AND 6
       OR CAST(geolocation_lng AS DOUBLE) NOT BETWEEN -74 AND -34
""")

# zips that disagree with themselves
show("""
    SELECT
        COUNT(*) FILTER (WHERE n_states > 1) AS zips_with_2plus_states,
        COUNT(*) FILTER (WHERE n_cities > 1) AS zips_with_2plus_cities
    FROM (
        SELECT geolocation_zip_code_prefix,
               COUNT(DISTINCT geolocation_state) AS n_states,
               COUNT(DISTINCT geolocation_city)  AS n_cities
        FROM raw.geolocation
        GROUP BY 1
    )
""")

# why the city count is inflated
show("SELECT DISTINCT geolocation_city FROM raw.geolocation WHERE geolocation_zip_code_prefix = '01310'")

┌─────────┬───────┬────────────────────┐
│  rows   │ zips  │ avg_points_per_zip │
│  int64  │ int64 │       double       │
├─────────┼───────┼────────────────────┤
│ 1000163 │ 19015 │               52.6 │
└─────────┴───────┴────────────────────┘

┌────────────────┐
│ duplicate_rows │
│     int128     │
├────────────────┤
│         261831 │
└────────────────┘

┌────────────────┐
│ outside_brazil │
│     int64      │
├────────────────┤
│             42 │
└────────────────┘

┌────────────────────────┬────────────────────────┐
│ zips_with_2plus_states │ zips_with_2plus_cities │
│         int64          │         int64          │
├────────────────────────┼────────────────────────┤
│                      8 │                   8556 │
└────────────────────────┴────────────────────────┘

┌──────────────────┐
│ geolocation_city │
│     varchar      │
├──────────────────┤
│ sao paulo        │
│ são paulo        │
└──────────────────┘



### Findings

| Issue | Size |
| --- | --- |
| Rows versus distinct zips | 1,000,163 rows across 19,015 zips - 52.6 points per zip on average, up to 1,146 |
| Exact duplicate rows | 261,831 (26% of the file) |
| Coordinates outside Brazil | 42 |
| Zips reporting more than one state | 8 |
| Zips reporting more than one city name | 8,556 |

The city inconsistency is spelling rather than geography: zip `01310` appears as both `sao paulo` and `são paulo`. Accents are applied inconsistently in this file and are absent entirely from the customers and sellers files, so any join on city text would fail silently.

### Decisions

1. **Drop the 42 out-of-Brazil points before aggregating.** A single point in the wrong hemisphere would move a zip's centroid by hundreds of kilometres.
2. **Collapse to one row per zip.** Joining this file at its native grain would multiply order rows roughly fifty-fold.
3. **Use the median rather than the mean** for latitude and longitude. The median absorbs a handful of mistyped coordinates inside an otherwise sound zip; the mean does not.
4. **Accept the loss of 5 zips.** Five of the 19,015 zips contain only out-of-Brazil coordinates, so filtering empties them. 19,010 remain.
5. **Take the most common city and state per zip** and strip accents, so the text matches the other files.
6. **Retain `n_points`** as a confidence measure: a centroid built from three points is weaker evidence than one built from three hundred, which matters when distances are reported.

Exact duplicate rows need no separate treatment - aggregating to one row per zip removes them by construction.

In [24]:
con.sql("""
    CREATE OR REPLACE TABLE clean.geo_zip AS
    SELECT
        geolocation_zip_code_prefix                   AS zip,
        MEDIAN(CAST(geolocation_lat AS DOUBLE))       AS lat,
        MEDIAN(CAST(geolocation_lng AS DOUBLE))       AS lng,
        MODE(geolocation_state)                       AS state,
        strip_accents(MODE(geolocation_city))         AS city,
        COUNT(*)                                      AS n_points
    FROM raw.geolocation
    WHERE CAST(geolocation_lat AS DOUBLE) BETWEEN -34 AND 6
      AND CAST(geolocation_lng AS DOUBLE) BETWEEN -74 AND -34
    GROUP BY 1
""")

check("clean.geo_zip rows", "SELECT COUNT(*) FROM clean.geo_zip", 19010)
check("zips lost with their bad points",
      "SELECT 19015 - COUNT(*) FROM clean.geo_zip", 5)
check("one row per zip",    "SELECT COUNT(*) - COUNT(DISTINCT zip) FROM clean.geo_zip", 0)
check("null coordinates",   "SELECT COUNT(*) FROM clean.geo_zip WHERE lat IS NULL OR lng IS NULL", 0)
show("SELECT * FROM clean.geo_zip ORDER BY n_points DESC LIMIT 5")

clean.geo_zip rows: 19,010 (expected 19,010) OK
zips lost with their bad points: 5 (expected 5) OK
one row per zip: 0 (expected 0) OK
null coordinates: 0 (expected 0) OK
┌─────────┬─────────────────────┬─────────────────────┬─────────┬─────────────┬──────────┐
│   zip   │         lat         │         lng         │  state  │    city     │ n_points │
│ varchar │       double        │       double        │ varchar │   varchar   │  int64   │
├─────────┼─────────────────────┼─────────────────────┼─────────┼─────────────┼──────────┤
│ 24220   │ -22.903166252842077 │  -43.10713068500763 │ RJ      │ niteroi     │     1146 │
│ 24230   │  -22.90600354640406 │  -43.10546782889948 │ RJ      │ niteroi     │     1102 │
│ 38400   │ -18.914980232586977 │  -48.27813616348312 │ MG      │ uberlandia  │      965 │
│ 35500   │  -20.14068469699654 │  -44.88753081910666 │ MG      │ divinopolis │      907 │
│ 11680   │ -23.441645336112845 │ -45.077753319947206 │ SP      │ ubatuba     │      879 │
└─────────┴

---

## 2. customers

**Grain:** one order-level customer record, not one person. `customer_id` is regenerated for every order; `customer_unique_id` is intended to identify the person.

Every repeat-purchase and lifetime-value question rests on `customer_unique_id` being trustworthy, so that is what the checks below test.

In [25]:
show("""
    SELECT
        COUNT(*)                            AS rows,
        COUNT(DISTINCT customer_id)         AS customer_ids,
        COUNT(DISTINCT customer_unique_id)  AS people,
        COUNT(DISTINCT customer_state)      AS states,
        COUNT(DISTINCT customer_city)       AS cities
    FROM raw.customers
""")

# does one "person" appear in several places?
show("""
    SELECT
        COUNT(*) FILTER (WHERE n_zips   > 1) AS people_with_2plus_zips,
        COUNT(*) FILTER (WHERE n_states > 1) AS people_with_2plus_states
    FROM (
        SELECT customer_unique_id,
               COUNT(DISTINCT customer_zip_code_prefix) AS n_zips,
               COUNT(DISTINCT customer_state)           AS n_states
        FROM raw.customers
        GROUP BY 1
    )
""")

# zip text integrity, and coverage against the lookup we just built
show("SELECT LENGTH(customer_zip_code_prefix) AS zip_length, COUNT(*) FROM raw.customers GROUP BY 1")
show("""
    SELECT COUNT(DISTINCT c.customer_zip_code_prefix) AS zips_without_coordinates
    FROM raw.customers c
    LEFT JOIN clean.geo_zip g ON g.zip = c.customer_zip_code_prefix
    WHERE g.zip IS NULL
""")

┌───────┬──────────────┬────────┬────────┬────────┐
│ rows  │ customer_ids │ people │ states │ cities │
│ int64 │    int64     │ int64  │ int64  │ int64  │
├───────┼──────────────┼────────┼────────┼────────┤
│ 99441 │        99441 │  96096 │     27 │   4119 │
└───────┴──────────────┴────────┴────────┴────────┘

┌────────────────────────┬──────────────────────────┐
│ people_with_2plus_zips │ people_with_2plus_states │
│         int64          │          int64           │
├────────────────────────┼──────────────────────────┤
│                    250 │                       39 │
└────────────────────────┴──────────────────────────┘

┌────────────┬──────────────┐
│ zip_length │ count_star() │
│   int64    │    int64     │
├────────────┼──────────────┤
│          5 │        99441 │
└────────────┴──────────────┘

┌──────────────────────────┐
│ zips_without_coordinates │
│          int64           │
├──────────────────────────┤
│                      158 │
└──────────────────────────┘



### Findings

| Issue | Size |
| --- | --- |
| Rows versus people | 99,441 `customer_id` values, 96,096 `customer_unique_id` values |
| People appearing in more than one zip | 250 |
| People appearing in more than one state | 39 |
| Zip codes stored as 5-digit text | all 99,441 - no leading zeros lost |
| Customer zips with no coordinates | 158, covering 279 orders |

The zip finding is worth recording: this file preserves leading zeros correctly, so the well-known Olist zip problem only appears if the CSV is read as numeric. Reading everything as text at load avoided it entirely - a trap avoided rather than a fix applied.

The 250 people appearing in multiple zips are a genuine limitation. A customer who moved would produce this pattern, but so would an identifier collision. At 0.3% of people it is small, so the identifier is kept and the caveat carried into any repeat-purchase claim.

### Decisions

1. **Keep all 99,441 rows** at their original grain.
2. **Attach coordinates from the zip lookup**, leaving them null where the zip is unknown rather than substituting a nearby one.
3. **Add `has_coordinates`**, so distance-based analysis excludes those rows explicitly instead of dropping them silently in a join.
4. **Take city and state from the zip lookup** where available, retaining the file's own values as `city_raw` and `state_raw` for comparison.

In [26]:
con.sql("""
    CREATE OR REPLACE TABLE clean.customers AS
    SELECT
        c.customer_id,
        c.customer_unique_id,
        c.customer_zip_code_prefix              AS zip,
        COALESCE(g.city,  c.customer_city)      AS city,
        COALESCE(g.state, c.customer_state)     AS state,
        c.customer_city                         AS city_raw,
        c.customer_state                        AS state_raw,
        g.lat,
        g.lng,
        (g.zip IS NOT NULL)                     AS has_coordinates
    FROM raw.customers c
    LEFT JOIN clean.geo_zip g ON g.zip = c.customer_zip_code_prefix
""")

check("clean.customers rows",      "SELECT COUNT(*) FROM clean.customers", 99441)
check("customer_id unique",        "SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM clean.customers", 0)
check("rows without coordinates",  "SELECT COUNT(*) FROM clean.customers WHERE NOT has_coordinates")
check("state disagrees with zip",  "SELECT COUNT(*) FROM clean.customers WHERE state <> state_raw")

clean.customers rows: 99,441 (expected 99,441) OK
customer_id unique: 0 (expected 0) OK
rows without coordinates: 279
state disagrees with zip: 0


---

## 3. sellers

**Grain:** one seller.

The seller forms one end of every delivery route, so this file drives both the distance calculation and all route-level analysis. Its location text is the least reliable in the dataset.

In [27]:
show("SELECT COUNT(*) AS rows, COUNT(DISTINCT seller_id) AS sellers, COUNT(DISTINCT seller_city) AS city_spellings, COUNT(DISTINCT seller_state) AS states FROM raw.sellers")

# city values that are clearly not city names
show("SELECT seller_id, seller_city, seller_state FROM raw.sellers WHERE seller_city ~ '[0-9@]' ORDER BY seller_id")

# city values carrying extra structure
show("""
    SELECT seller_city, COUNT(*) AS n
    FROM raw.sellers
    WHERE seller_city LIKE '%/%' OR seller_city LIKE '%-%'
       OR seller_city LIKE '%,%' OR seller_city LIKE '%  %'
    GROUP BY 1 ORDER BY 2 DESC, 1
""", n=30)

# does the stated state agree with the zip?
show("""
    SELECT COUNT(*) AS state_mismatches
    FROM raw.sellers s
    JOIN clean.geo_zip g ON g.zip = s.seller_zip_code_prefix
    WHERE g.state <> s.seller_state
""")

show("""
    SELECT s.seller_state AS stated, g.state AS from_zip, COUNT(*) AS n
    FROM raw.sellers s
    JOIN clean.geo_zip g ON g.zip = s.seller_zip_code_prefix
    WHERE g.state <> s.seller_state
    GROUP BY 1,2 ORDER BY 3 DESC, 1, 2 LIMIT 10
""")

┌───────┬─────────┬────────────────┬────────┐
│ rows  │ sellers │ city_spellings │ states │
│ int64 │  int64  │     int64      │ int64  │
├───────┼─────────┼────────────────┼────────┤
│  3095 │    3095 │            611 │     23 │
└───────┴─────────┴────────────────┴────────┘

┌──────────────────────────────────┬───────────────────────────┬──────────────┐
│            seller_id             │        seller_city        │ seller_state │
│             varchar              │          varchar          │   varchar    │
├──────────────────────────────────┼───────────────────────────┼──────────────┤
│ 4b5f66b7adcf57f1ecc0d3c07dd6b177 │ vendas@creditparts.com.br │ PR           │
│ ceb7b4fb9401cd378de7886317ad1b47 │ 04482255                  │ RJ           │
└──────────────────────────────────┴───────────────────────────┴──────────────┘

┌──────────────────────────────────────────┬───────┐
│               seller_city                │   n   │
│                 varchar                  │ int64 │
├──

### Findings

| Issue | Size |
| --- | --- |
| City values that are not cities | 2 - `vendas@creditparts.com.br` and `04482255` |
| City values containing separators or double spaces | 25, such as `sao paulo / sao paulo`, `lages - sc`, `sbc/sp` |
| Distinct city spellings | 611 for 3,095 sellers |
| State contradicting the zip prefix | 35 sellers |
| Zips with no coordinates | 7 |

This is free text entered by sellers themselves, which is why one of them is an email address.

### Decisions

1. **Replace the city text rather than clean it.** Reconciling 611 spellings by hand is neither reliable nor reproducible. The zip prefix is a structured field that resolves to a known place, so city and state are taken from the zip lookup and the typed values retained as `city_raw` and `state_raw`. This resolves the 2 invalid values, the 25 malformed ones and the 35 contradicted states in a single step, and reduces 611 spellings to 555 canonical ones.
2. **Flag `state_mismatch` rather than overwrite silently.** If route-level results later look implausible, these 35 sellers can be tested as a cause.
3. **Fall back to the typed state** for the 7 sellers whose zip is absent from the lookup.

Using a structured column to repair an unstructured one is more defensible than any sequence of string fixes, and it is reproducible when the data is refreshed.

In [28]:
con.sql("""
    CREATE OR REPLACE TABLE clean.sellers AS
    SELECT
        s.seller_id,
        s.seller_zip_code_prefix                        AS zip,
        COALESCE(g.city,  s.seller_city)                AS city,
        COALESCE(g.state, s.seller_state)               AS state,
        s.seller_city                                   AS city_raw,
        s.seller_state                                  AS state_raw,
        g.lat,
        g.lng,
        (g.zip IS NOT NULL)                             AS has_coordinates,
        (g.state IS NOT NULL AND g.state <> s.seller_state) AS state_mismatch
    FROM raw.sellers s
    LEFT JOIN clean.geo_zip g ON g.zip = s.seller_zip_code_prefix
""")

check("clean.sellers rows",         "SELECT COUNT(*) FROM clean.sellers", 3095)
check("seller_id unique",           "SELECT COUNT(*) - COUNT(DISTINCT seller_id) FROM clean.sellers", 0)
check("without coordinates",        "SELECT COUNT(*) FROM clean.sellers WHERE NOT has_coordinates")
check("state corrected",            "SELECT COUNT(*) FROM clean.sellers WHERE state_mismatch")
check("city spellings now",         "SELECT COUNT(DISTINCT city) FROM clean.sellers")
show("SELECT city_raw, city, state_raw, state FROM clean.sellers WHERE city_raw ~ '[0-9@]' OR state_mismatch ORDER BY (city_raw ~ '[0-9@]') DESC, city_raw LIMIT 8")

clean.sellers rows: 3,095 (expected 3,095) OK
seller_id unique: 0 (expected 0) OK
without coordinates: 7
state corrected: 35
city spellings now: 555
┌───────────────────────────┬────────────────┬───────────┬─────────┐
│         city_raw          │      city      │ state_raw │  state  │
│          varchar          │    varchar     │  varchar  │ varchar │
├───────────────────────────┼────────────────┼───────────┼─────────┤
│ 04482255                  │ rio de janeiro │ RJ        │ RJ      │
│ vendas@creditparts.com.br │ maringa        │ PR        │ PR      │
│ aguas claras df           │ brasilia       │ SP        │ DF      │
│ andradas                  │ andradas       │ SP        │ MG      │
│ belo horizonte            │ belo horizonte │ SP        │ MG      │
│ belo horizonte            │ belo horizonte │ SP        │ MG      │
│ blumenau                  │ blumenau       │ SP        │ SC      │
│ caxias do sul             │ caxias do sul  │ SP        │ RS      │
└──────────────────────

---

## 4. category_translation

**Grain:** one product category, Portuguese to English.

Category is one of the primary dimensions for every finding in the project. A category that fails to translate becomes a null, and nulls disappear from charts without announcing themselves.

In [29]:
show("SELECT COUNT(*) AS rows FROM raw.category_translation")

# categories used by products but missing from the lookup
show("""
    SELECT p.product_category_name, COUNT(*) AS products
    FROM raw.products p
    LEFT JOIN raw.category_translation t USING (product_category_name)
    WHERE p.product_category_name IS NOT NULL
      AND t.product_category_name IS NULL
    GROUP BY 1
""")

# misspellings on the English side
show("""
    SELECT product_category_name, product_category_name_english
    FROM raw.category_translation
    WHERE product_category_name_english LIKE '%costruction%'
       OR product_category_name_english LIKE '%confort%'
       OR product_category_name_english LIKE 'fashio!_%' ESCAPE '!'
""")

┌───────┐
│ rows  │
│ int64 │
├───────┤
│    71 │
└───────┘

┌───────────────────────────────────────────────┬──────────┐
│             product_category_name             │ products │
│                    varchar                    │  int64   │
├───────────────────────────────────────────────┼──────────┤
│ portateis_cozinha_e_preparadores_de_alimentos │       10 │
│ pc_gamer                                      │        3 │
└───────────────────────────────────────────────┴──────────┘

┌────────────────────────────────────┬───────────────────────────────┐
│       product_category_name        │ product_category_name_english │
│              varchar               │            varchar            │
├────────────────────────────────────┼───────────────────────────────┤
│ construcao_ferramentas_jardim      │ costruction_tools_garden      │
│ casa_conforto                      │ home_confort                  │
│ construcao_ferramentas_ferramentas │ costruction_tools_tools       │
│ fashion_roup

### Findings

| Issue | Size |
| --- | --- |
| Categories used by products but absent from the file | 2 - `pc_gamer` (3 products), `portateis_cozinha_e_preparadores_de_alimentos` (10 products) |
| Misspelled English names | 4 - `costruction_tools_garden`, `costruction_tools_tools`, `home_confort`, `fashio_female_clothing` |

Only 13 products are affected by the missing rows, but they would surface as "unknown" on a category chart, which reads as a defect to anyone looking at the dashboard.

### Decisions

1. **Add the two missing categories** with translations written here. They are mine rather than Olist's, and are marked as such in the code.
2. **Correct the four misspelled English names.** These become axis labels on the dashboard, so they should read properly.
3. **Join on the Portuguese name only.** The English column is the one being edited, so it is never used as a key.

In [30]:
con.sql("""
    CREATE OR REPLACE TABLE clean.category_translation AS
    SELECT
        product_category_name,
        replace(replace(replace(product_category_name_english,
            'costruction', 'construction'),
            'confort',     'comfort'),
            'fashio_',     'fashion_')                 AS category_en
    FROM raw.category_translation

    UNION ALL

    -- added by hand: used by products but absent from Olist's file
    SELECT * FROM (VALUES
        ('pc_gamer',                                      'pc_gamer'),
        ('portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_food_processors')
    ) AS t(product_category_name, category_en)
""")

check("clean.category_translation rows", "SELECT COUNT(*) FROM clean.category_translation", 73)
check("category key unique",             "SELECT COUNT(*) - COUNT(DISTINCT product_category_name) FROM clean.category_translation", 0)
check("categories still untranslated",   """
    SELECT COUNT(DISTINCT p.product_category_name) FROM raw.products p
    LEFT JOIN clean.category_translation t USING (product_category_name)
    WHERE p.product_category_name IS NOT NULL AND t.product_category_name IS NULL""", 0)
show("SELECT * FROM clean.category_translation WHERE category_en LIKE 'construction%' OR category_en LIKE '%comfort%' OR category_en LIKE 'fashion_f%'")

clean.category_translation rows: 73 (expected 73) OK
category key unique: 0 (expected 0) OK
categories still untranslated: 0 (expected 0) OK
┌────────────────────────────────────┬─────────────────────────────────┐
│       product_category_name        │           category_en           │
│              varchar               │             varchar             │
├────────────────────────────────────┼─────────────────────────────────┤
│ construcao_ferramentas_construcao  │ construction_tools_construction │
│ construcao_ferramentas_jardim      │ construction_tools_garden       │
│ casa_conforto                      │ home_comfort                    │
│ construcao_ferramentas_ferramentas │ construction_tools_tools        │
│ construcao_ferramentas_iluminacao  │ construction_tools_lights       │
│ construcao_ferramentas_seguranca   │ construction_tools_safety       │
│ fashion_roupa_feminina             │ fashion_female_clothing         │
│ casa_conforto_2                    │ home_comfort_2   

---

## 5. products

**Grain:** one product.

The analysis needs two things from this file: category, for slicing results, and weight and dimensions, because freight is priced on them and the freight-fairness analysis depends on them.

In [31]:
show("""
    SELECT
        COUNT(*)                                                  AS rows,
        COUNT(DISTINCT product_id)                                AS products,
        COUNT(*) FILTER (WHERE product_category_name IS NULL)     AS no_category,
        COUNT(*) FILTER (WHERE product_weight_g IS NULL)          AS no_weight,
        COUNT(*) FILTER (WHERE CAST(product_weight_g AS DOUBLE) = 0) AS zero_weight,
        COUNT(*) FILTER (WHERE product_length_cm IS NULL)         AS no_dimensions
    FROM raw.products
""")

show("SELECT column_name FROM information_schema.columns WHERE table_schema = 'raw' AND table_name = 'products'")

show("""
    SELECT
        MIN(CAST(product_weight_g AS DOUBLE)) AS min_weight_g,
        MAX(CAST(product_weight_g AS DOUBLE)) AS max_weight_g,
        MAX(CAST(product_length_cm AS DOUBLE) * CAST(product_height_cm AS DOUBLE)
            * CAST(product_width_cm AS DOUBLE)) AS max_volume_cm3
    FROM raw.products
""")

┌───────┬──────────┬─────────────┬───────────┬─────────────┬───────────────┐
│ rows  │ products │ no_category │ no_weight │ zero_weight │ no_dimensions │
│ int64 │  int64   │    int64    │   int64   │    int64    │     int64     │
├───────┼──────────┼─────────────┼───────────┼─────────────┼───────────────┤
│ 32951 │    32951 │         610 │         2 │           4 │             2 │
└───────┴──────────┴─────────────┴───────────┴─────────────┴───────────────┘

┌────────────────────────────┐
│        column_name         │
│          varchar           │
├────────────────────────────┤
│ product_id                 │
│ product_category_name      │
│ product_name_lenght        │
│ product_description_lenght │
│ product_photos_qty         │
│ product_weight_g           │
│ product_length_cm          │
│ product_height_cm          │
│ product_width_cm           │
└────────────────────────────┘

┌──────────────┬──────────────┬────────────────┐
│ min_weight_g │ max_weight_g │ max_volume_cm3 │
│   

### Findings

| Issue | Size |
| --- | --- |
| Products with no category | 610 of 32,951 |
| Products with no weight or dimensions | 2 |
| Products with a weight of zero | 4 |
| Misspelled column names | `product_name_lenght`, `product_description_lenght` |

A weight of zero grams is not a measurement, it is a missing value recorded as a number. Left in place it lowers any average weight and distorts the freight model - the kind of error that survives into a dashboard because nothing visibly breaks.

### Decisions

1. **Convert zero weights to null.** A null is excluded from averages automatically; a zero is not.
2. **Label missing categories `unknown`** rather than leaving them null, so they appear as a visible bar instead of vanishing from charts.
3. **Compute `volume_cm3` once here** rather than repeating a three-column multiplication in every later query.
4. **Correct the two misspelled column names.**
5. **Add `has_dimensions`** so the freight model can filter cleanly.

In [32]:
con.sql("""
    CREATE OR REPLACE TABLE clean.products AS
    SELECT
        p.product_id,
        p.product_category_name                              AS category_pt,
        COALESCE(t.category_en, 'unknown')                   AS category,
        NULLIF(CAST(p.product_weight_g AS DOUBLE), 0)        AS weight_g,
        CAST(p.product_length_cm AS DOUBLE)                  AS length_cm,
        CAST(p.product_height_cm AS DOUBLE)                  AS height_cm,
        CAST(p.product_width_cm  AS DOUBLE)                  AS width_cm,
        CAST(p.product_length_cm AS DOUBLE)
          * CAST(p.product_height_cm AS DOUBLE)
          * CAST(p.product_width_cm AS DOUBLE)               AS volume_cm3,
        CAST(p.product_photos_qty AS INT)                    AS photos_qty,
        CAST(p.product_name_lenght AS INT)                   AS name_length,
        CAST(p.product_description_lenght AS INT)            AS description_length,
        (p.product_length_cm IS NOT NULL
         AND NULLIF(CAST(p.product_weight_g AS DOUBLE), 0) IS NOT NULL) AS has_dimensions
    FROM raw.products p
    LEFT JOIN clean.category_translation t USING (product_category_name)
""")

check("clean.products rows",   "SELECT COUNT(*) FROM clean.products", 32951)
check("product_id unique",     "SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM clean.products", 0)
check("category 'unknown'",    "SELECT COUNT(*) FROM clean.products WHERE category = 'unknown'", 610)
check("weight now null",       "SELECT COUNT(*) FROM clean.products WHERE weight_g IS NULL", 6)
check("usable for freight",    "SELECT COUNT(*) FROM clean.products WHERE has_dimensions")
show("SELECT category, COUNT(*) AS products FROM clean.products GROUP BY 1 ORDER BY 2 DESC LIMIT 8")

clean.products rows: 32,951 (expected 32,951) OK
product_id unique: 0 (expected 0) OK
category 'unknown': 610 (expected 610) OK
weight now null: 6 (expected 6) OK
usable for freight: 32,945
┌───────────────────────┬──────────┐
│       category        │ products │
│        varchar        │  int64   │
├───────────────────────┼──────────┤
│ bed_bath_table        │     3029 │
│ sports_leisure        │     2867 │
│ furniture_decor       │     2657 │
│ health_beauty         │     2444 │
│ housewares            │     2335 │
│ auto                  │     1900 │
│ computers_accessories │     1639 │
│ toys                  │     1411 │
└───────────────────────┴──────────┘



---

## 6. orders

**Grain:** one order. This is the spine of the project - every delivery metric is the difference between two of its five timestamps.

| Column | Event |
| --- | --- |
| `order_purchase_timestamp` | customer places the order |
| `order_approved_at` | payment clears |
| `order_delivered_carrier_date` | seller hands the parcel to the carrier |
| `order_delivered_customer_date` | parcel reaches the customer |
| `order_estimated_delivery_date` | the date promised at checkout |

Two things need testing: whether these events actually occur in that order, and whether the promised date is comparable to the delivered date.

In [33]:
show("SELECT order_status, COUNT(*) AS n FROM raw.orders GROUP BY 1 ORDER BY 2 DESC")

# missing dates, and the contradictions between status and dates
show("""
    SELECT
        COUNT(*) FILTER (WHERE order_approved_at IS NULL)             AS no_approval,
        COUNT(*) FILTER (WHERE order_delivered_carrier_date IS NULL)  AS no_carrier,
        COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS no_delivery,
        COUNT(*) FILTER (WHERE order_status = 'delivered'
                           AND order_delivered_customer_date IS NULL) AS delivered_but_no_date,
        COUNT(*) FILTER (WHERE order_status <> 'delivered'
                           AND order_delivered_customer_date IS NOT NULL) AS not_delivered_but_dated
    FROM raw.orders
""")

# do the events happen in a possible order?
show("""
    SELECT
        COUNT(*) FILTER (WHERE CAST(order_approved_at AS TIMESTAMP)
                             < CAST(order_purchase_timestamp AS TIMESTAMP))        AS approved_before_purchase,
        COUNT(*) FILTER (WHERE CAST(order_delivered_carrier_date AS TIMESTAMP)
                             < CAST(order_approved_at AS TIMESTAMP))               AS carrier_before_approval,
        COUNT(*) FILTER (WHERE CAST(order_delivered_customer_date AS TIMESTAMP)
                             < CAST(order_delivered_carrier_date AS TIMESTAMP))    AS delivered_before_carrier
    FROM raw.orders
""")

# THE IMPORTANT ONE: does the promised date carry a time of day?
show("""
    SELECT
        COUNT(*) FILTER (WHERE order_estimated_delivery_date LIKE '% 00:00:00')     AS estimated_at_midnight,
        COUNT(*) FILTER (WHERE order_purchase_timestamp NOT LIKE '% 00:00:00')      AS purchase_has_real_time,
        COUNT(*) FILTER (WHERE order_delivered_customer_date NOT LIKE '% 00:00:00') AS delivery_has_real_time
    FROM raw.orders
""")

┌──────────────┬───────┐
│ order_status │   n   │
│   varchar    │ int64 │
├──────────────┼───────┤
│ delivered    │ 96478 │
│ shipped      │  1107 │
│ canceled     │   625 │
│ unavailable  │   609 │
│ invoiced     │   314 │
│ processing   │   301 │
│ created      │     5 │
│ approved     │     2 │
└──────────────┴───────┘

┌─────────────┬────────────┬─────────────┬───────────────────────┬─────────────────────────┐
│ no_approval │ no_carrier │ no_delivery │ delivered_but_no_date │ not_delivered_but_dated │
│    int64    │   int64    │    int64    │         int64         │          int64          │
├─────────────┼────────────┼─────────────┼───────────────────────┼─────────────────────────┤
│         160 │       1783 │        2965 │                     8 │                       6 │
└─────────────┴────────────┴─────────────┴───────────────────────┴─────────────────────────┘

┌──────────────────────────┬─────────────────────────┬──────────────────────────┐
│ approved_before_purchase │ carr

### Findings

| Issue | Size |
| --- | --- |
| Orders with no delivery date | 2,965 - mostly undelivered, which is correct |
| Marked `delivered` with no delivery date | 8 |
| Marked `canceled` but carrying a delivery date | 6 |
| Carrier pickup recorded before payment approval | 1,359 |
| Delivery recorded before carrier pickup | 23 |
| `order_estimated_delivery_date` at exactly 00:00:00 | all 99,441 |

**The promised date has no time of day.** Every estimate is stored at midnight while real deliveries carry real clock times. Comparing the two as timestamps therefore counts an order delivered at 14:00 on the promised day as fourteen hours late. This affects 1,292 orders and moves the headline late rate from 6.77% to 8.11% - a 20% relative difference in the project's most important single figure.

### Decisions

1. **Measure the promise in whole days, date to date.** `days_vs_promise` is the difference between the delivered date and the promised date, so delivery on the promised day is 0 rather than late. Olist promised a day, not a moment, and the measurement should match the promise.
2. **Measure durations with full timestamps.** `delivery_days` keeps fractional precision, since purchase and delivery both carry real clock times. Promise comparison uses dates; elapsed time uses timestamps. These are different questions and are treated differently.
3. **Flag the 1,382 impossible sequences as `timestamp_anomaly` and keep them.** They distort the dispatch-versus-transit split, which is the only place they need excluding; their lateness and review data remains valid.
4. **Keep all 99,441 rows.** Restricting to delivered orders is a scope decision for the fact table, not a cleaning decision, and cancelled and unavailable orders are needed to reconcile payments with revenue and to count orders that never arrived.
5. **Derive `is_delivered`** so that no later query has to reinterpret eight status strings.

In [34]:
con.sql("""
    CREATE OR REPLACE TABLE clean.orders AS
    WITH typed AS (
        SELECT
            order_id,
            customer_id,
            order_status                                        AS status,
            CAST(order_purchase_timestamp      AS TIMESTAMP)    AS purchase_ts,
            CAST(order_approved_at             AS TIMESTAMP)    AS approved_ts,
            CAST(order_delivered_carrier_date  AS TIMESTAMP)    AS carrier_ts,
            CAST(order_delivered_customer_date AS TIMESTAMP)    AS delivered_ts,
            CAST(order_estimated_delivery_date AS DATE)         AS promised_date
        FROM raw.orders
    )
    SELECT
        *,
        CAST(delivered_ts AS DATE)                              AS delivered_date,
        -- promise: whole days, compared date to date
        date_diff('day', promised_date, CAST(delivered_ts AS DATE)) AS days_vs_promise,
        date_diff('day', promised_date, CAST(delivered_ts AS DATE)) > 0 AS is_late,
        -- durations: full timestamps, fractional days
        date_diff('second', purchase_ts, delivered_ts) / 86400.0 AS delivery_days,
        date_diff('second', purchase_ts, approved_ts)  / 86400.0 AS approval_days,
        date_diff('second', approved_ts, carrier_ts)   / 86400.0 AS dispatch_days,
        date_diff('second', carrier_ts,  delivered_ts) / 86400.0 AS transit_days,
        -- data-quality flags
        (approved_ts IS NULL OR carrier_ts IS NULL
         OR carrier_ts   < approved_ts
         OR delivered_ts < carrier_ts)                          AS timestamp_anomaly,
        (status = 'delivered' AND delivered_ts IS NOT NULL)     AS is_delivered
    FROM typed
""")

check("clean.orders rows",        "SELECT COUNT(*) FROM clean.orders", 99441)
check("order_id unique",          "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM clean.orders", 0)
check("delivered in scope",       "SELECT COUNT(*) FROM clean.orders WHERE is_delivered")
check("timestamp anomalies",      "SELECT COUNT(*) FROM clean.orders WHERE is_delivered AND timestamp_anomaly")
check("negative delivery_days",   "SELECT COUNT(*) FROM clean.orders WHERE delivery_days < 0", 0)

# the two ways of measuring lateness, side by side
show("""
    SELECT
        ROUND(100 * AVG(CASE WHEN is_late THEN 1.0 ELSE 0 END), 2)          AS late_rate_by_date,
        ROUND(100 * AVG(CASE WHEN delivered_ts > promised_date THEN 1.0 ELSE 0 END), 2) AS late_rate_by_timestamp,
        COUNT(*) FILTER (WHERE delivered_date = promised_date)              AS delivered_on_the_promised_day
    FROM clean.orders WHERE is_delivered
""")

clean.orders rows: 99,441 (expected 99,441) OK
order_id unique: 0 (expected 0) OK
delivered in scope: 96,470
timestamp anomalies: 1,388
negative delivery_days: 0 (expected 0) OK
┌───────────────────┬────────────────────────┬───────────────────────────────┐
│ late_rate_by_date │ late_rate_by_timestamp │ delivered_on_the_promised_day │
│      double       │         double         │             int64             │
├───────────────────┼────────────────────────┼───────────────────────────────┤
│              6.77 │                   8.11 │                          1292 │
└───────────────────┴────────────────────────┴───────────────────────────────┘



---

## 7. order_items

**Grain:** one item within an order - 112,650 rows against 99,441 orders.

This is the table that breaks order-level figures: joined to `orders` without aggregating first, every order-level measure is multiplied by its item count. The checks below establish the real key, the plausibility of prices and freight, and whether any item points at a product, seller or order that does not exist.

In [35]:
show("""
    SELECT
        COUNT(*)                                        AS rows,
        COUNT(DISTINCT order_id)                        AS orders,
        COUNT(*) - COUNT(DISTINCT order_id || '|' || order_item_id) AS duplicate_keys,
        MAX(CAST(order_item_id AS INT))                 AS max_items_in_an_order
    FROM raw.order_items
""")

show("""
    SELECT
        MIN(CAST(price AS DOUBLE))                                   AS min_price,
        MAX(CAST(price AS DOUBLE))                                   AS max_price,
        COUNT(*) FILTER (WHERE CAST(price AS DOUBLE) <= 0)           AS price_zero_or_less,
        COUNT(*) FILTER (WHERE CAST(freight_value AS DOUBLE) = 0)    AS freight_zero,
        COUNT(*) FILTER (WHERE CAST(freight_value AS DOUBLE) < 0)    AS freight_negative
    FROM raw.order_items
""")

# referential integrity in all three directions
show("""
    SELECT
        COUNT(*) FILTER (WHERE o.order_id   IS NULL) AS orphan_order,
        COUNT(*) FILTER (WHERE p.product_id IS NULL) AS orphan_product,
        COUNT(*) FILTER (WHERE s.seller_id  IS NULL) AS orphan_seller
    FROM raw.order_items i
    LEFT JOIN raw.orders    o USING (order_id)
    LEFT JOIN raw.products  p USING (product_id)
    LEFT JOIN raw.sellers   s USING (seller_id)
""")

# how often does one order involve more than one seller?
show("""
    SELECT COUNT(*) AS multi_seller_orders
    FROM (SELECT order_id FROM raw.order_items GROUP BY 1 HAVING COUNT(DISTINCT seller_id) > 1)
""")

┌────────┬────────┬────────────────┬───────────────────────┐
│  rows  │ orders │ duplicate_keys │ max_items_in_an_order │
│ int64  │ int64  │     int64      │         int32         │
├────────┼────────┼────────────────┼───────────────────────┤
│ 112650 │  98666 │              0 │                    21 │
└────────┴────────┴────────────────┴───────────────────────┘

┌───────────┬───────────┬────────────────────┬──────────────┬──────────────────┐
│ min_price │ max_price │ price_zero_or_less │ freight_zero │ freight_negative │
│  double   │  double   │       int64        │    int64     │      int64       │
├───────────┼───────────┼────────────────────┼──────────────┼──────────────────┤
│      0.85 │    6735.0 │                  0 │          383 │                0 │
└───────────┴───────────┴────────────────────┴──────────────┴──────────────────┘

┌──────────────┬────────────────┬───────────────┐
│ orphan_order │ orphan_product │ orphan_seller │
│    int64     │     int64      │     int64   

### Findings

| Check | Result |
| --- | --- |
| Key | `order_id` + `order_item_id` is unique; `order_id` alone is not |
| Items per order | up to 21 |
| Orders involving more than one seller | 1,278 |
| Prices | 0.85 to 6,735 BRL, none zero or negative |
| Freight of exactly zero | 383 items |
| Orphan rows | none - every item resolves to a real order, product and seller |

This table is in good condition. The single judgement call is the 383 items with zero freight. Negative freight would indicate an error; zero is plausible as free shipping, and these rows sit alongside normal prices.

### Decisions

1. **Keep the item grain.** Aggregating to one row per order happens in the modelling notebook where the choice of aggregation is explicit. Doing it here would conceal it.
2. **Keep the zero-freight items and flag them `free_shipping`.** If the freight model behaves oddly, these 383 rows can be tested as a cause.
3. **Cast prices to `DOUBLE` and the shipping limit to a timestamp**, so that later arithmetic cannot silently compare text.
4. **Carry `seller_id` and `product_id` through unchanged** - they are clean keys and the dimensions attach to them in the next notebook.

In [36]:
con.sql("""
    CREATE OR REPLACE TABLE clean.order_items AS
    SELECT
        order_id,
        CAST(order_item_id AS INT)                  AS item_number,
        product_id,
        seller_id,
        CAST(shipping_limit_date AS TIMESTAMP)      AS shipping_limit_ts,
        CAST(price AS DOUBLE)                       AS price,
        CAST(freight_value AS DOUBLE)               AS freight,
        (CAST(freight_value AS DOUBLE) = 0)         AS free_shipping
    FROM raw.order_items
""")

check("clean.order_items rows", "SELECT COUNT(*) FROM clean.order_items", 112650)
check("key is unique",          "SELECT COUNT(*) - COUNT(DISTINCT order_id || '|' || item_number) FROM clean.order_items", 0)
check("free shipping items",    "SELECT COUNT(*) FROM clean.order_items WHERE free_shipping", 383)
check("total item revenue",     "SELECT ROUND(SUM(price), 2) FROM clean.order_items")

clean.order_items rows: 112,650 (expected 112,650) OK
key is unique: 0 (expected 0) OK
free shipping items: 383 (expected 383) OK
total item revenue: 13591643.7


---

## 8. order_payments

**Grain:** one payment line. An order may be settled with several vouchers plus a card, so one order can carry many rows - up to 29.

The checks below cover payment types, the plausibility of values and instalment counts, and whether payments reconcile to what the items cost.

In [37]:
show("SELECT payment_type, COUNT(*) AS n FROM raw.order_payments GROUP BY 1 ORDER BY 2 DESC")

show("""
    SELECT
        COUNT(*)                                                        AS rows,
        COUNT(DISTINCT order_id)                                        AS orders,
        MAX(CAST(payment_sequential AS INT))                            AS max_lines_on_one_order,
        COUNT(*) FILTER (WHERE CAST(payment_value AS DOUBLE) = 0)       AS value_zero,
        COUNT(*) FILTER (WHERE CAST(payment_installments AS INT) = 0)   AS installments_zero,
        MAX(CAST(payment_installments AS INT))                          AS max_installments
    FROM raw.order_payments
""")

# do payments reconcile with the items?
show("""
    SELECT COUNT(*) AS orders_off_by_more_than_1_brl
    FROM      (SELECT order_id, SUM(CAST(payment_value AS DOUBLE)) AS paid  FROM raw.order_payments GROUP BY 1) p
    JOIN      (SELECT order_id, SUM(CAST(price AS DOUBLE) + CAST(freight_value AS DOUBLE)) AS owed
               FROM raw.order_items GROUP BY 1) i USING (order_id)
    WHERE ABS(paid - owed) > 1
""")

┌──────────────┬───────┐
│ payment_type │   n   │
│   varchar    │ int64 │
├──────────────┼───────┤
│ credit_card  │ 76795 │
│ boleto       │ 19784 │
│ voucher      │  5775 │
│ debit_card   │  1529 │
│ not_defined  │     3 │
└──────────────┴───────┘

┌────────┬────────┬────────────────────────┬────────────┬───────────────────┬──────────────────┐
│  rows  │ orders │ max_lines_on_one_order │ value_zero │ installments_zero │ max_installments │
│ int64  │ int64  │         int32          │   int64    │       int64       │      int32       │
├────────┼────────┼────────────────────────┼────────────┼───────────────────┼──────────────────┤
│ 103886 │  99440 │                     29 │          9 │                 2 │               24 │
└────────┴────────┴────────────────────────┴────────────┴───────────────────┴──────────────────┘

┌───────────────────────────────┐
│ orders_off_by_more_than_1_brl │
│             int64             │
├───────────────────────────────┤
│                           24

### Findings

| Issue | Size |
| --- | --- |
| Payment types | credit_card 76,795, boleto 19,784, voucher 5,775, debit_card 1,529, `not_defined` 3 |
| Payment lines with a value of zero | 9 |
| Instalment count of zero | 2 |
| Orders where payments differ from item cost by more than 1 BRL | 249 |

`not_defined` is a placeholder rather than a payment method, and an instalment count of zero is impossible - the minimum real value is a single payment.

### Two revenue figures, and why they differ

Summing `payment_value` gives 16,008,872 BRL. Summing item price plus freight gives 15,843,553 BRL. The 165,319 BRL difference resolves exactly:

| Cause | Orders | BRL |
| --- | --- | --- |
| Paid for, but the order carries no item lines | 775 | +162,592 |
| Credit-card instalment interest | 232 | +3,065 |
| Paid less than the items cost | 17 | −198 |
| Sub-1 BRL rounding across all remaining orders | 98,416 | +3 |
| One order with items but no payment record | 1 | −143 |
| **Total** | | **+165,319** |

The dominant cause is orders that were paid for but never fulfilled: of those 775, 603 are `unavailable` and 164 are `canceled`. The payment record survives; the item lines do not.

The second is instalment interest. Under Brazilian *parcelamento com juros*, paying across instalments adds interest, and the gap scales with the instalment count - roughly +13% at ten instalments, +24% at twenty, +29% at twenty-four. Olist recorded the amount the customer was charged, which includes money owed to the card issuer rather than to the marketplace.

### Decisions

1. **Keep the payment-line grain**, aggregating in the modelling notebook.
2. **Set `not_defined` to null**, so it cannot be mistaken for a real category on a chart.
3. **Raise instalment counts of 0 to 1.**
4. **Define revenue as `order_items.price` throughout.** Instalment interest belongs to the card issuer, payments on unfulfilled orders are not sales, and price and freight must stay separable for the freight analysis. Payment value fuses the two. Restricting the fact table to delivered orders removes the largest discrepancy by construction, leaving 249 orders out of roughly 96,000.

In [38]:
con.sql("""
    CREATE OR REPLACE TABLE clean.payments AS
    SELECT
        order_id,
        CAST(payment_sequential AS INT)                     AS payment_line,
        NULLIF(payment_type, 'not_defined')                 AS payment_type,
        GREATEST(CAST(payment_installments AS INT), 1)      AS installments,
        CAST(payment_value AS DOUBLE)                       AS payment_value
    FROM raw.order_payments
""")

check("clean.payments rows",      "SELECT COUNT(*) FROM clean.payments", 103886)
check("key unique",               "SELECT COUNT(*) - COUNT(DISTINCT order_id || '|' || payment_line) FROM clean.payments", 0)
check("payment_type now null",    "SELECT COUNT(*) FROM clean.payments WHERE payment_type IS NULL", 3)
check("installments below 1",     "SELECT COUNT(*) FROM clean.payments WHERE installments < 1", 0)

# H1 from the brief: is payment approval a real stage, or minutes?
show("""
    SELECT
        p.payment_type,
        COUNT(*)                                        AS orders,
        ROUND(MEDIAN(o.approval_days * 24), 2)          AS median_hours,
        ROUND(QUANTILE_CONT(o.approval_days * 24, 0.9), 2) AS p90_hours
    FROM clean.orders o
    JOIN (SELECT order_id, ARG_MAX(payment_type, payment_value) AS payment_type
          FROM clean.payments GROUP BY 1) p USING (order_id)
    WHERE o.approved_ts IS NOT NULL
    GROUP BY 1 ORDER BY 2 DESC
""")

clean.payments rows: 103,886 (expected 103,886) OK
key unique: 0 (expected 0) OK
payment_type now null: 3 (expected 3) OK
installments below 1: 0 (expected 0) OK
┌──────────────┬────────┬──────────────┬───────────┐
│ payment_type │ orders │ median_hours │ p90_hours │
│   varchar    │ int64  │    double    │  double   │
├──────────────┼────────┼──────────────┼───────────┤
│ credit_card  │  74921 │         0.27 │     21.39 │
│ boleto       │  19754 │        29.07 │     67.27 │
│ voucher      │   3078 │         0.28 │     24.22 │
│ debit_card   │   1527 │         0.85 │     24.04 │
└──────────────┴────────┴──────────────┴───────────┘



---

## 9. order_reviews

**Grain:** nominally one review per order. It is not, and this file carries the most consequential problem in the dataset.

Two separate things need testing: whether the grain is really one review per order, and *when* each review was written relative to the delivery it describes.

In [39]:
show("""
    SELECT
        COUNT(*)                        AS rows,
        COUNT(DISTINCT review_id)       AS distinct_review_ids,
        COUNT(DISTINCT order_id)        AS distinct_orders
    FROM raw.order_reviews
""")

show("""
    SELECT
        COUNT(*) FILTER (WHERE n > 1) AS orders_with_several_reviews
    FROM (SELECT order_id, COUNT(*) AS n FROM raw.order_reviews GROUP BY 1)
""")

show("""
    SELECT COUNT(*) AS orders_whose_reviews_disagree
    FROM (SELECT order_id FROM raw.order_reviews
          GROUP BY 1 HAVING COUNT(*) > 1 AND COUNT(DISTINCT review_score) > 1)
""")

show("SELECT review_score, COUNT(*) AS n FROM raw.order_reviews GROUP BY 1 ORDER BY 1")

# THE KEY QUESTION: was the review written before the parcel arrived?
show("""
    SELECT
        o.is_late,
        COUNT(*)                                                                    AS reviewed_orders,
        COUNT(*) FILTER (WHERE CAST(r.review_creation_date AS TIMESTAMP) < o.delivered_ts) AS written_before_delivery,
        ROUND(100.0 * AVG(CASE WHEN CAST(r.review_creation_date AS TIMESTAMP) < o.delivered_ts
                               THEN 1 ELSE 0 END), 1)                               AS pct_before_delivery,
        ROUND(AVG(CAST(r.review_score AS DOUBLE)), 2)                               AS avg_score
    FROM raw.order_reviews r
    JOIN clean.orders o USING (order_id)
    WHERE o.is_delivered
    GROUP BY 1
""")

# and does it change the score?
show("""
    SELECT
        CAST(r.review_creation_date AS TIMESTAMP) < o.delivered_ts AS written_before_delivery,
        COUNT(*)                                      AS late_orders,
        ROUND(AVG(CAST(r.review_score AS DOUBLE)), 2) AS avg_score
    FROM raw.order_reviews r
    JOIN clean.orders o USING (order_id)
    WHERE o.is_delivered AND o.is_late
    GROUP BY 1
""")

┌───────┬─────────────────────┬─────────────────┐
│ rows  │ distinct_review_ids │ distinct_orders │
│ int64 │        int64        │      int64      │
├───────┼─────────────────────┼─────────────────┤
│ 99224 │               98410 │           98673 │
└───────┴─────────────────────┴─────────────────┘

┌─────────────────────────────┐
│ orders_with_several_reviews │
│            int64            │
├─────────────────────────────┤
│                         547 │
└─────────────────────────────┘

┌───────────────────────────────┐
│ orders_whose_reviews_disagree │
│             int64             │
├───────────────────────────────┤
│                           202 │
└───────────────────────────────┘

┌──────────────┬───────┐
│ review_score │   n   │
│   varchar    │ int64 │
├──────────────┼───────┤
│ 1            │ 11424 │
│ 2            │  3151 │
│ 3            │  8179 │
│ 4            │ 19142 │
│ 5            │ 57328 │
└──────────────┴───────┘

┌─────────┬─────────────────┬─────────────────────

### Findings

| Issue | Size |
| --- | --- |
| Rows versus orders | 99,224 rows, 98,673 orders, 98,410 distinct `review_id` |
| Orders carrying more than one review | 547 |
| Those whose two reviews give different scores | 202 |
| Score distribution | strongly bimodal: 57,328 fives against 11,424 ones |

**Review timing.** On late orders, 83.6% of reviews were created *before the parcel arrived*. On on-time orders the figure is 3.3%. Olist sent its review survey on the estimated delivery date, so when an order runs late the customer is asked to rate an order they have not yet received.

The effect on the score is large. Among late orders, reviews written before the parcel arrived average 1.98 stars; those written after it arrived average 3.78.

The conventional reading of this dataset - that late delivery destroys review scores - is therefore partly a description of when the survey was sent. The defensible version is narrower and more useful: being asked to rate an order that has not arrived produces a very low score. Every review figure in this project is consequently reported split by whether the review preceded or followed delivery.

### Decisions

1. **Deduplicate to one review per order**, keeping the most recently answered. The latest answer represents the customer's final position, and 202 orders carry genuinely conflicting scores, so an explicit rule is required.
2. **Retain `review_id`** so duplicates remain traceable.
3. **Compute the before/after delivery flag in the modelling notebook**, not here. It requires `clean.orders`, and clean tables stay single-source.
4. **Keep `has_comment` rather than the comment text.** 58,247 reviews carry no message, and the text itself is only needed later for complaint classification.

In [40]:
con.sql("""
    CREATE OR REPLACE TABLE clean.reviews AS
    SELECT order_id, review_id, review_score, review_creation_ts, review_answer_ts,
           has_comment, n_reviews_on_order
    FROM (
        SELECT
            order_id,
            review_id,
            CAST(review_score AS INT)                       AS review_score,
            CAST(review_creation_date AS TIMESTAMP)         AS review_creation_ts,
            CAST(review_answer_timestamp AS TIMESTAMP)      AS review_answer_ts,
            (review_comment_message IS NOT NULL)            AS has_comment,
            COUNT(*)      OVER (PARTITION BY order_id)      AS n_reviews_on_order,
            ROW_NUMBER()  OVER (PARTITION BY order_id
                                ORDER BY CAST(review_answer_timestamp AS TIMESTAMP) DESC,
                                         review_id)         AS rn
        FROM raw.order_reviews
    )
    WHERE rn = 1
""")

check("clean.reviews rows",    "SELECT COUNT(*) FROM clean.reviews", 98673)
check("one review per order",  "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM clean.reviews", 0)
check("rows dropped as dupes", "SELECT 99224 - COUNT(*) FROM clean.reviews", 551)
check("score out of range",    "SELECT COUNT(*) FROM clean.reviews WHERE review_score NOT BETWEEN 1 AND 5", 0)
show("SELECT review_score, COUNT(*) AS n FROM clean.reviews GROUP BY 1 ORDER BY 1")

clean.reviews rows: 98,673 (expected 98,673) OK
one review per order: 0 (expected 0) OK
rows dropped as dupes: 551 (expected 551) OK
score out of range: 0 (expected 0) OK
┌──────────────┬───────┐
│ review_score │   n   │
│    int32     │ int64 │
├──────────────┼───────┤
│            1 │ 11363 │
│            2 │  3131 │
│            3 │  8133 │
│            4 │ 19038 │
│            5 │ 57008 │
└──────────────┴───────┘



---

## Verification

Three questions close this stage: does every clean table exist at the grain claimed for it, did any row disappear that should not have, and does revenue still reconcile to the raw file?

In [41]:
expected_rows = {
    "geo_zip":              19010,
    "customers":            99441,
    "sellers":               3095,
    "category_translation":    73,
    "products":             32951,
    "orders":               99441,
    "order_items":         112650,
    "payments":            103886,
    "reviews":              98673,
}

print("ROW COUNTS")
for t, n in expected_rows.items():
    got = con.execute(f"SELECT COUNT(*) FROM clean.{t}").fetchall()[0][0]
    print(f"  clean.{t:22}{got:>9,}  expected {n:>9,}  {'OK' if got == n else 'MISMATCH'}")

print("\nGRAIN (0 means the key is unique)")
for t, key in [("geo_zip", "zip"), ("customers", "customer_id"), ("sellers", "seller_id"),
               ("category_translation", "product_category_name"), ("products", "product_id"),
               ("orders", "order_id"), ("reviews", "order_id")]:
    dupes = con.execute(
        f"SELECT COUNT(*) - COUNT(DISTINCT {key}) FROM clean.{t}").fetchall()[0][0]
    print(f"  clean.{t:22}{key:28}{dupes}")

print("\nRECONCILIATION")
raw_rev   = con.execute("SELECT ROUND(SUM(CAST(price AS DOUBLE)), 2) FROM raw.order_items").fetchall()[0][0]
clean_rev = con.execute("SELECT ROUND(SUM(price), 2) FROM clean.order_items").fetchall()[0][0]
print(f"  item revenue raw {raw_rev:,.2f} vs clean {clean_rev:,.2f}  "
      f"{'OK' if abs(raw_rev - clean_rev) < 0.01 else 'MISMATCH'}")

print("\nROWS INTENTIONALLY REMOVED")
print(f"  reviews deduplicated : {99224 - 98673} rows (several reviews on one order)")
geo_rows = con.execute("SELECT COUNT(*) FROM clean.geo_zip").fetchall()[0][0]
print(f"  geolocation collapsed: {1000163 - geo_rows:,} rows (many points per zip -> one centroid)")

ROW COUNTS
  clean.geo_zip                  19,010  expected    19,010  OK
  clean.customers                99,441  expected    99,441  OK
  clean.sellers                   3,095  expected     3,095  OK
  clean.category_translation         73  expected        73  OK
  clean.products                 32,951  expected    32,951  OK
  clean.orders                   99,441  expected    99,441  OK
  clean.order_items             112,650  expected   112,650  OK
  clean.payments                103,886  expected   103,886  OK
  clean.reviews                  98,673  expected    98,673  OK

GRAIN (0 means the key is unique)
  clean.geo_zip               zip                         0
  clean.customers             customer_id                 0
  clean.sellers               seller_id                   0
  clean.category_translation  product_category_name       0
  clean.products              product_id                  0
  clean.orders                order_id                    0
  clean.reviews   

In [42]:
# release the file lock so 03_data_model.ipynb can open the database
con.close()
print("connection closed")

connection closed


---

## Decision log

Every issue found during cleaning and the decision taken on it.

| # | Table | Issue | Size | Decision |
| --- | --- | --- | --- | --- |
| 1 | geolocation | Many points per zip, 26% exact duplicates | 1,000,163 rows to 19,010 zips | One row per zip, median coordinates |
| 2 | geolocation | Coordinates outside Brazil | 42 points | Dropped before aggregating |
| 3 | geolocation | Accented and unaccented city names within one zip | 8,556 zips | Most common spelling, accents stripped |
| 4 | customers | One `customer_unique_id` across several zips | 250 people | Retained; stated as a caveat on repeat-purchase results |
| 5 | customers | Zips without coordinates | 158 zips, 279 orders | Retained, flagged `has_coordinates` |
| 6 | sellers | City field containing an email address and a number | 2 sellers | City and state derived from the zip lookup |
| 7 | sellers | 611 city spellings, 25 containing separators | 3,095 sellers | Same: derived from zip, typed values kept as `_raw` |
| 8 | sellers | State contradicting the zip prefix | 35 sellers | Zip takes precedence; flagged `state_mismatch` |
| 9 | category_translation | Two categories absent | 13 products | Added, marked as my own translation |
| 10 | category_translation | Misspelled English names | 4 categories | Corrected - they become dashboard labels |
| 11 | products | No category | 610 products | Labelled `unknown` to stay visible |
| 12 | products | Weight recorded as 0 g | 4 products | Converted to null |
| 13 | products | Misspelled column names (`lenght`) | 2 columns | Renamed |
| 14 | orders | Promised date carries no time of day | all rows | Promise measured in whole days by date; late rate 6.77% rather than 8.11% |
| 15 | orders | Carrier pickup before payment approval | 1,359 orders | Flagged `timestamp_anomaly`, excluded only from the stage split |
| 16 | orders | Delivery before carrier pickup | 23 orders | Same flag |
| 17 | orders | Marked delivered with no delivery date | 8 orders | Excluded by `is_delivered` |
| 18 | order_items | Freight of exactly zero | 383 items | Retained, flagged `free_shipping` |
| 19 | payments | `not_defined` payment type | 3 rows | Set to null |
| 20 | payments | Instalment count of zero | 2 rows | Raised to 1 |
| 21 | payments | Payments not matching item cost | 249 orders, 165,319 BRL in aggregate | Revenue defined as `order_items.price`; difference reconciled in section 8 |
| 22 | reviews | Several reviews on one order | 547 orders, 202 conflicting | Kept the most recently answered |
| 23 | reviews | Reviews written before delivery | 83.6% of late orders | Not a cleaning fix: review figures are reported split by before/after delivery |

The cleaned tables are the input to `03_data_model.ipynb`, which builds the order-level fact table and the dimensions around it.